In [1]:
import os
import gc
import copy
import sys
import random
import torch
import torch.nn as nn
import numpy as np
import pandas as pd

from collections        import defaultdict

from pathlib            import Path

from torch.utils.data   import (DataLoader,
                                Subset,
                                WeightedRandomSampler)

from sklearn.metrics    import (accuracy_score,
                                confusion_matrix,
                                classification_report,
                                balanced_accuracy_score,
                                f1_score)

sys.path.append('..')
import module

from src.model.modules  import (MicroExpressionDataset,
                                Normalizer,
                                SpatialTemporalCNN)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
torch.backends.cudnn.benchmark = True
print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')

Device: cuda
PyTorch: 2.10.0+cu128


In [2]:
# ── Konfigurasi ────────────────────────────────────────────────────────
ANNOTATIONS  = os.path.join(Path.home(), "datasets", "anxiety_raw", "annotations-v2.csv")
STAGE        = 'all'      # before + after
N_SPLITS     = 5          # GroupKFold
SEED         = 42

EPOCHS       = 200
LR           = 3e-4
WEIGHT_DECAY = 1e-4
BATCH_SIZE   = 32

# Early stopping patience per strategi
PATIENCE_A   = 20    # Strategi A: stop dari train loss
PATIENCE_B   = 20    # Strategi B: stop dari val UAR
PATIENCE_C   = 20    # Strategi C: stop dari train loss (checkpoint dari val UAR)

# Direktori checkpoint
CKPT_DIR     = './checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

print('Konfigurasi:')
print(f'  Annotations : {ANNOTATIONS}')
print(f'  Stage       : {STAGE}')
print(f'  GroupKFold  : {N_SPLITS} fold')
print(f'  Epochs      : {EPOCHS}')
print(f'  LR          : {LR}')
print(f'  Batch size  : {BATCH_SIZE}')
print(f'  Checkpoint  : {CKPT_DIR}')

Konfigurasi:
  Annotations : /home/inadio/datasets/anxiety_raw/annotations-v2.csv
  Stage       : all
  GroupKFold  : 5 fold
  Epochs      : 200
  LR          : 0.0003
  Batch size  : 32
  Checkpoint  : ./checkpoints


In [3]:
# ── Dataset ─────────────────────────────────────────────────────────────
dataset = MicroExpressionDataset(
    annotations_file = ANNOTATIONS,
    stage            = STAGE,
    mode             = 'train',
    valid_only       = True,
)

print(f'Total samples : {len(dataset)}')
print(f'Total subjek  : {dataset.annotations["subject_id"].nunique()}')
y_all = dataset.get_labels()
print(f'Label dist    : high={( y_all==1).sum()}, low={(y_all==0).sum()}')

# ── GroupKFold ────────────────────────────────────────────────────────
folds = MicroExpressionDataset.build_group_kfold(
    dataset.annotations, n_splits=N_SPLITS)

print(f'\nGroupKFold-{N_SPLITS} siap.')

Total samples : 559
Total subjek  : 56
Label dist    : high=265, low=294

GroupKFold-5 siap.


In [4]:
# ── Helper: build DataLoader dengan WeightedRandomSampler ───────────────
def make_loaders(dataset, train_idx, test_idx, batch_size):
    train_labels = dataset.get_labels()[train_idx]
    class_counts = np.bincount(train_labels, minlength=2)
    weights      = torch.tensor(
        [1.0 / class_counts[l] for l in train_labels], dtype=torch.float)
    sampler      = WeightedRandomSampler(
        weights, len(weights), replacement=True)

    train_loader = DataLoader(
        Subset(dataset, train_idx),
        batch_size  = batch_size,
        sampler     = sampler,
        num_workers = 0,
        pin_memory  = True,
        drop_last   = True,
    )
    test_loader = DataLoader(
        Subset(dataset, test_idx),
        batch_size  = batch_size * 2,
        shuffle     = False,
        num_workers = 0,
        pin_memory  = True,
    )
    return train_loader, test_loader


# ── Helper: normalisasi fold (fit hanya pada training) ───────────────────
def normalize_fold(dataset, train_idx, test_idx):
    """
    Ekstrak semua fitur dalam mode eval (tanpa augmentasi),
    fit Normalizer pada training set, transform keduanya.
    Mengembalikan X_train, X_test, y_train, y_test yang sudah dinormalisasi.
    """
    dataset.set_mode('eval')
    dataset.clear_cache()

    X_all, y_all, subs_all = dataset.get_all_features()

    X_train = X_all[train_idx]
    y_train = y_all[train_idx]
    X_test  = X_all[test_idx]
    y_test  = y_all[test_idx]

    norm    = Normalizer()
    X_train = norm.fit_transform(X_train)
    X_test  = norm.transform(X_test)

    return X_train, X_test, y_train, y_test, norm


# ── Helper: buat DataLoader dari numpy array (setelah normalisasi) ───────
from torch.utils.data import TensorDataset

def make_tensor_loaders(X_train, y_train, X_test, y_test, batch_size,
                         subs_train, subs_test):
    """
    Buat DataLoader dari numpy array yang sudah dinormalisasi.
    Training loader pakai WeightedRandomSampler.
    """
    # Bungkus dalam TensorDataset
    # Tambahkan subject sebagai integer index (untuk evaluate)
    subj_train_t = torch.arange(len(X_train))  # placeholder
    subj_test_t  = torch.arange(len(X_test))

    tr_ds = TensorDataset(
        torch.from_numpy(X_train),
        torch.from_numpy(y_train).long(),
        subj_train_t,
    )
    te_ds = TensorDataset(
        torch.from_numpy(X_test),
        torch.from_numpy(y_test).long(),
        subj_test_t,
    )

    class_counts = np.bincount(y_train, minlength=2)
    weights      = torch.tensor(
        [1.0 / class_counts[l] for l in y_train], dtype=torch.float)
    sampler      = WeightedRandomSampler(weights, len(weights), replacement=True)

    train_loader = DataLoader(tr_ds, batch_size=batch_size,
                              sampler=sampler, drop_last=True)
    test_loader  = DataLoader(te_ds, batch_size=batch_size * 2, shuffle=False)

    return train_loader, test_loader


# ── Train one epoch ──────────────────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    total_loss = 0.0
    for feat, label, _ in loader:
        feat  = feat.to(device, non_blocking=True)
        label = label.to(device, non_blocking=True)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            pred = model(feat)
            loss = criterion(pred, label)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    return total_loss / max(len(loader), 1)


# ── Evaluate ─────────────────────────────────────────────────────────────
def evaluate(model, loader, subject_names, verbose=False):
    """
    Evaluasi di level clip dan level subject.
    subject_names : list[str] panjang = len(dataset test), mapping index -> subject_id
    """
    model.eval()
    sample_preds, sample_targets = [], []
    subject_probs   = defaultdict(list)
    subject_targets = {}

    with torch.no_grad():
        for feat, label, idx in loader:
            feat = feat.to(device, non_blocking=True)
            out  = model(feat)
            prob = torch.softmax(out, dim=1).cpu().numpy()
            pred = np.argmax(prob, axis=1)
            lnp  = label.numpy()

            sample_preds.extend(pred.tolist())
            sample_targets.extend(lnp.tolist())

            for i, si in enumerate(idx.tolist()):
                subj = subject_names[si]
                subject_probs[subj].append(prob[i])
                subject_targets[subj] = int(lnp[i])

    subj_names = sorted(subject_probs.keys())
    subj_preds = [int(np.argmax(np.mean(subject_probs[s], axis=0)))
                  for s in subj_names]
    subj_true  = [subject_targets[s] for s in subj_names]

    m = {
        'sample_targets'          : sample_targets,
        'sample_preds'            : sample_preds,
        'sample_uar'              : balanced_accuracy_score(sample_targets, sample_preds),
        'subject_accuracy'        : accuracy_score(subj_true, subj_preds),
        'subject_uar'             : balanced_accuracy_score(subj_true, subj_preds),
        'subject_macro_f1'        : f1_score(subj_true, subj_preds,
                                             average='macro', zero_division=0),
        'subject_confusion_matrix': confusion_matrix(subj_true, subj_preds, labels=[0,1]),
        'subject_report'          : classification_report(
            subj_true, subj_preds, labels=[0,1],
            target_names=['low','high'], digits=4, zero_division=0),
    }

    if verbose:
        print(f'  sample UAR  : {m["sample_uar"]:.4f}')
        print(f'  subject acc : {m["subject_accuracy"]:.4f}')
        print(f'  subject UAR : {m["subject_uar"]:.4f}')
        print(f'  macro-F1    : {m["subject_macro_f1"]:.4f}')
        print(f'  confusion   :\n{m["subject_confusion_matrix"]}')
        print(f'  report      :\n{m["subject_report"]}')
    return m


# ── Build fresh model + optimizer + scheduler ────────────────────────────
def build_model():
    m   = SpatialTemporalCNN().to(device)
    opt = torch.optim.Adam(m.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt, T_max=EPOCHS, eta_min=1e-6)
    scl = torch.amp.GradScaler('cuda')
    return m, opt, sch, scl


print('Helper functions siap.')

Helper functions siap.


In [5]:
# ── Training Loop: GroupKFold-5 dengan 3 Strategi Early Stopping ────────
#
# Setiap fold menjalankan 3 model secara paralel:
#   Model A — stop: train loss    | ckpt: train loss terbaik
#   Model B — stop: val UAR       | ckpt: val UAR terbaik
#   Model C — stop: train loss    | ckpt: val UAR terbaik
#
# Output per fold:
#   - Metrik final ketiga model
#   - Checkpoint model terbaik global per strategi
# ─────────────────────────────────────────────────────────────────────────

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

# Akumulator hasil per strategi
results = {'A': [], 'B': [], 'C': []}
global_true  = {'A': [], 'B': [], 'C': []}
global_pred  = {'A': [], 'B': [], 'C': []}
best_global  = {'A': -1., 'B': -1., 'C': -1.}

for fold_info in folds:
    fold      = fold_info['fold']
    train_idx = fold_info['train_idx']
    test_idx  = fold_info['test_idx']

    print(f'\n{"="*65}')
    print(f'Fold {fold:02d} | '
          f'train={len(train_idx)} dist={fold_info["train_label_dist"]} | '
          f'test={len(test_idx)}  dist={fold_info["test_label_dist"]}')

    # ── Normalisasi (fit hanya pada training fold) ──────────────────
    X_train, X_test, y_train, y_test, norm = normalize_fold(
        dataset, train_idx, test_idx)

    # Subject name untuk evaluasi subject-level
    subj_all   = dataset.get_subject_ids()
    subj_train = [subj_all[i] for i in train_idx]
    subj_test  = [subj_all[i] for i in test_idx]

    train_loader, test_loader = make_tensor_loaders(
        X_train, y_train, X_test, y_test, BATCH_SIZE,
        subj_train, subj_test)

    # ── Inisialisasi 3 model identik (berbeda hanya di stopping logic) ─
    mA, optA, schA, sclA = build_model()
    mB, optB, schB, sclB = build_model()
    mC, optC, schC, sclC = build_model()

    # State tracking per strategi
    # Strategi A
    best_loss_A    = float('inf')
    ckpt_A         = None
    no_imp_A       = 0
    stopped_A      = False
    stopped_ep_A   = EPOCHS - 1

    # Strategi B
    best_uar_B     = -1.
    ckpt_B         = None
    no_imp_B       = 0
    stopped_B      = False
    stopped_ep_B   = EPOCHS - 1

    # Strategi C
    best_loss_C    = float('inf')
    best_uar_C     = -1.
    ckpt_C         = None
    no_imp_C       = 0
    stopped_C      = False
    stopped_ep_C   = EPOCHS - 1

    dataset.set_mode('train')

    for epoch in range(EPOCHS):
        any_active = not (stopped_A and stopped_B and stopped_C)
        if not any_active:
            break

        # ── Train satu epoch per model yang masih aktif ──────────────
        if not stopped_A:
            lossA = train_one_epoch(mA, train_loader, optA, criterion, sclA)
            schA.step()
        if not stopped_B:
            lossB = train_one_epoch(mB, train_loader, optB, criterion, sclB)
            schB.step()
        if not stopped_C:
            lossC = train_one_epoch(mC, train_loader, optC, criterion, sclC)
            schC.step()

        # ── Evaluasi pada test fold ───────────────────────────────────
        mA_uar = evaluate(mA, test_loader, subj_test)['subject_uar'] if not stopped_A else best_uar_B
        mB_uar = evaluate(mB, test_loader, subj_test)['subject_uar'] if not stopped_B else best_uar_B
        mC_uar = evaluate(mC, test_loader, subj_test)['subject_uar'] if not stopped_C else best_uar_C

        lr_now = schA.get_last_lr()[0] if not stopped_A else 0.

        # ── Update state Strategi A (stop & ckpt dari train loss) ─────
        if not stopped_A:
            if lossA < best_loss_A - 1e-4:
                best_loss_A = lossA
                no_imp_A    = 0
                ckpt_A      = {k: v.cpu().clone()
                               for k, v in mA.state_dict().items()}
            else:
                no_imp_A += 1
            if no_imp_A >= PATIENCE_A:
                stopped_A    = True
                stopped_ep_A = epoch
                print(f'  [A] Early stop ep {epoch} (train_loss)')

        # ── Update state Strategi B (stop & ckpt dari val UAR) ────────
        if not stopped_B:
            if mB_uar > best_uar_B:
                best_uar_B = mB_uar
                no_imp_B   = 0
                ckpt_B     = {k: v.cpu().clone()
                              for k, v in mB.state_dict().items()}
            else:
                no_imp_B += 1
            if no_imp_B >= PATIENCE_B:
                stopped_B    = True
                stopped_ep_B = epoch
                print(f'  [B] Early stop ep {epoch} (val UAR)')

        # ── Update state Strategi C (stop: loss | ckpt: UAR) ──────────
        if not stopped_C:
            if mC_uar > best_uar_C:   # checkpoint dari val UAR
                best_uar_C = mC_uar
                ckpt_C     = {k: v.cpu().clone()
                              for k, v in mC.state_dict().items()}
            if lossC < best_loss_C - 1e-4:   # stop dari train loss
                best_loss_C = lossC
                no_imp_C    = 0
            else:
                no_imp_C += 1
            if no_imp_C >= PATIENCE_C:
                stopped_C    = True
                stopped_ep_C = epoch
                print(f'  [C] Early stop ep {epoch} (train_loss, ckpt UAR)')

        # ── Log setiap 5 epoch ────────────────────────────────────────
        if epoch % 5 == 0 or (stopped_A and stopped_B and stopped_C):
            print(
                f'  Ep {epoch:03d} | lr {lr_now:.1e} | '
                f'A: loss={lossA if not stopped_A else 0:.4f} '
                f'uar={mA_uar:.3f} no_imp={no_imp_A}/{PATIENCE_A} | '
                f'B: loss={lossB if not stopped_B else 0:.4f} '
                f'uar={mB_uar:.3f} no_imp={no_imp_B}/{PATIENCE_B} | '
                f'C: loss={lossC if not stopped_C else 0:.4f} '
                f'uar={mC_uar:.3f} no_imp={no_imp_C}/{PATIENCE_C}'
            )

    # ── Evaluasi final tiap strategi ──────────────────────────────────
    print(f'\nFold {fold:02d} — Hasil Final:')
    fold_row = {'fold': fold,
                'test_subjects': sorted(fold_info['test_subjects']),
                'n_test': len(test_idx)}

    for tag, model, ckpt, ep_stopped in [
        ('A', mA, ckpt_A, stopped_ep_A),
        ('B', mB, ckpt_B, stopped_ep_B),
        ('C', mC, ckpt_C, stopped_ep_C),
    ]:
        if ckpt is not None:
            model.load_state_dict(ckpt)
        m = evaluate(model, test_loader, subj_test, verbose=False)

        print(f'  [{tag}] stopped=ep{ep_stopped:03d} | '
              f'UAR={m["subject_uar"]:.4f} | '
              f'F1={m["subject_macro_f1"]:.4f} | '
              f'Acc={m["subject_accuracy"]:.4f}')
        print(f'       confusion: {m["subject_confusion_matrix"].tolist()}')

        fold_row[f'uar_{tag}']  = m['subject_uar']
        fold_row[f'f1_{tag}']   = m['subject_macro_f1']
        fold_row[f'acc_{tag}']  = m['subject_accuracy']
        fold_row[f'ep_{tag}']   = ep_stopped

        global_true[tag].extend(m['sample_targets'])
        global_pred[tag].extend(m['sample_preds'])

        # Simpan checkpoint terbaik global
        if m['subject_uar'] > best_global[tag]:
            best_global[tag] = m['subject_uar']
            ckpt_path = os.path.join(
                CKPT_DIR,
                f'best_strategy{tag}_fold{fold:02d}'
                f'_uar{m["subject_uar"]:.4f}.pt')
            torch.save(ckpt, ckpt_path)
            # Save normalizer alongside checkpoint — required for inference
            norm_path = ckpt_path.replace('.pt', '_normalizer.npz')
            norm.save(norm_path)
            print(f'       >> Checkpoint saved: {ckpt_path}')
            print(f'       >> Normalizer saved: {norm_path}')

    results['A'].append(fold_row)
    results['B'].append(fold_row)
    results['C'].append(fold_row)

    # Cleanup
    del mA, mB, mC, optA, optB, optC
    del ckpt_A, ckpt_B, ckpt_C
    del X_train, X_test, y_train, y_test
    del train_loader, test_loader
    gc.collect()
    torch.cuda.empty_cache()


Fold 00 | train=440 dist=[200 240] | test=119  dist=[94 25]
  Ep 000 | lr 3.0e-04 | A: loss=1.1362 uar=0.773 no_imp=0/20 | B: loss=1.2327 uar=0.409 no_imp=0/20 | C: loss=1.4847 uar=0.500 no_imp=0/20
  Ep 005 | lr 3.0e-04 | A: loss=1.0970 uar=0.727 no_imp=3/20 | B: loss=0.9894 uar=0.273 no_imp=5/20 | C: loss=1.0732 uar=0.318 no_imp=0/20
  Ep 010 | lr 3.0e-04 | A: loss=0.8470 uar=0.727 no_imp=0/20 | B: loss=0.9441 uar=0.091 no_imp=10/20 | C: loss=0.9463 uar=0.636 no_imp=3/20
  Ep 015 | lr 3.0e-04 | A: loss=0.8355 uar=0.818 no_imp=2/20 | B: loss=0.8301 uar=0.091 no_imp=15/20 | C: loss=0.7955 uar=0.591 no_imp=2/20
  [B] Early stop ep 20 (val UAR)
  Ep 020 | lr 2.9e-04 | A: loss=0.7430 uar=0.818 no_imp=0/20 | B: loss=0.0000 uar=0.091 no_imp=20/20 | C: loss=0.8249 uar=0.364 no_imp=1/20
  Ep 025 | lr 2.9e-04 | A: loss=0.6587 uar=0.727 no_imp=0/20 | B: loss=0.0000 uar=0.409 no_imp=20/20 | C: loss=0.7388 uar=0.636 no_imp=2/20
  Ep 030 | lr 2.8e-04 | A: loss=0.6824 uar=0.682 no_imp=5/20 | B: lo

In [6]:
# ── Ringkasan Perbandingan 3 Strategi ───────────────────────────────────
print(f'\n{"="*65}')
print('RINGKASAN GroupKFold-5 — Perbandingan Early Stopping')
print(f'{"="*65}')

df_res = pd.DataFrame(results['A'])  # fold_row sama untuk ketiga strategi

for tag in ['A', 'B', 'C']:
    label = {
        'A': 'Strategi A (stop: train_loss | ckpt: train_loss)',
        'B': 'Strategi B (stop: val_UAR    | ckpt: val_UAR)',
        'C': 'Strategi C (stop: train_loss | ckpt: val_UAR)',
    }[tag]

    uars = df_res[f'uar_{tag}'].values
    f1s  = df_res[f'f1_{tag}'].values
    eps  = df_res[f'ep_{tag}'].values

    # Clip-level global
    g_uar = balanced_accuracy_score(global_true[tag], global_pred[tag])
    g_f1  = f1_score(global_true[tag], global_pred[tag],
                     average='macro', zero_division=0)
    g_cm  = confusion_matrix(global_true[tag], global_pred[tag], labels=[0,1])

    print(f'\n{label}')
    print(f'  Subject UAR (mean +/- std) : {uars.mean():.4f} +/- {uars.std():.4f}')
    print(f'  Subject F1  (mean +/- std) : {f1s.mean():.4f}  +/- {f1s.std():.4f}')
    print(f'  Clip UAR global            : {g_uar:.4f}')
    print(f'  Clip F1  global            : {g_f1:.4f}')
    print(f'  Avg epochs stopped         : {eps.mean():.1f}')
    print(f'  Confusion (global)         : {g_cm.tolist()}')

# Tabel per fold
print(f'\nPer-fold UAR per strategi:')
header = f'{"fold":>5} {"A_uar":>7} {"B_uar":>7} {"C_uar":>7} '\
         f'{"A_ep":>6} {"B_ep":>6} {"C_ep":>6}'
print(header)
print('-' * len(header))
for _, row in df_res.iterrows():
    print(f'{int(row["fold"]):>5} '
          f'{row["uar_A"]:>7.4f} {row["uar_B"]:>7.4f} {row["uar_C"]:>7.4f} '
          f'{int(row["ep_A"]):>6} {int(row["ep_B"]):>6} {int(row["ep_C"]):>6}')

# Bootstrap 95% CI untuk strategi terbaik
print(f'\nBootstrap 95% CI Subject UAR (n=5000):')
for tag in ['A', 'B', 'C']:
    uars   = df_res[f'uar_{tag}'].values
    boots  = [np.mean(np.random.choice(uars, len(uars), replace=True))
              for _ in range(5000)]
    lo, hi = np.percentile(boots, [2.5, 97.5])
    print(f'  Strategi {tag}: [{lo:.4f}, {hi:.4f}]')


RINGKASAN GroupKFold-5 — Perbandingan Early Stopping

Strategi A (stop: train_loss | ckpt: train_loss)
  Subject UAR (mean +/- std) : 0.4926 +/- 0.2011
  Subject F1  (mean +/- std) : 0.4319  +/- 0.1131
  Clip UAR global            : 0.4634
  Clip F1  global            : 0.4634
  Avg epochs stopped         : 154.0
  Confusion (global)         : [[146, 148], [151, 114]]

Strategi B (stop: val_UAR    | ckpt: val_UAR)
  Subject UAR (mean +/- std) : 0.6091 +/- 0.1296
  Subject F1  (mean +/- std) : 0.5180  +/- 0.1355
  Clip UAR global            : 0.5086
  Clip F1  global            : 0.5063
  Avg epochs stopped         : 29.0
  Confusion (global)         : [[177, 117], [155, 110]]

Strategi C (stop: train_loss | ckpt: val_UAR)
  Subject UAR (mean +/- std) : 0.6896 +/- 0.1421
  Subject F1  (mean +/- std) : 0.5819  +/- 0.1491
  Clip UAR global            : 0.5044
  Clip F1  global            : 0.5043
  Avg epochs stopped         : 150.2
  Confusion (global)         : [[159, 135], [141, 124]]